In [0]:

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import avg, count, round, col

In [0]:
# Azure Event Hubs Connection String, Event Hub Namespace, and name

CONNECTION_STR = ""

EVENTHUB_NAME = "evnt_hub_store"

EVENTHUB_NS = 'evnt-hub-stream-ns'

connectionConf = {
  'eventhubs.connectionString' : spark.conf.set('spark.eventhubs.connectionString', CONNECTION_STR),
  'eventhubs.name': EVENTHUB_NAME
}

In [0]:

# Default storage location for catalog
storage_location = 'abfss://unity-catalog-storage@dbstoragejyjel7oyffces.dfs.core.windows.net/7405614095555477'

# Create Unity Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS weather_data_stream MANAGED LOCATION '{storage_location}'")

# Create bronze layer schema
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_data_stream.bronze")

# Create silver layer schema
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_data_stream.silver")

# Create gold layer schema
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_data_stream.gold")

# Create bronze layer volume for checkpoint
spark.sql("CREATE VOLUME IF NOT EXISTS weather_data_stream.bronze.bronze_volume")

# Create silver layer volume for checkpoint
spark.sql("CREATE VOLUME IF NOT EXISTS weather_data_stream.silver.silver_volume")

# Create gold layer volume for checkpoint
spark.sql("CREATE VOLUME IF  NOT EXISTS weather_data_stream.gold.gold_volume")

DataFrame[]

In [0]:
df_bronze = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{EVENTHUB_NS}.servicebus.windows.net:9093")
    .option("subscribe", f"{EVENTHUB_NAME}")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.ssl.endpoint.identification.algorithm", "https")
    .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{CONNECTION_STR}";')
    .load())


In [0]:
# dbutils.fs.rm("/Volumes/weather_data_stream/bronze/bronze_volume/stream/", recurse=True)

# checkpoint_path = "/Volumes/weather_data_stream/bronze/bronze_volume/stream/"
# display(df_bronze, checkpointLocation=checkpoint_path)

In [0]:
import time

run_id = int(time.time())
checkpoint_path = f"/Volumes/weather_data_stream/bronze/bronze_volume/stream/path_{run_id}"
table_name = "weather_data_stream.bronze.bronze_data"



# Start the stream to the table
query = df_bronze.writeStream \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .format("delta") \
    .toTable(table_name)




In [0]:

# see what is actually landing in your Bronze layer
# display(spark.readStream.table(table_name), checkpointLocation=display_checkpoint)

In [0]:
%sql
SELECT * FROM weather_data_stream.bronze.bronze_data

key,value,topic,partition,offset,timestamp,timestampType
null,eyJjaXR5IjogIkNoaWNhZ28iLCAibGF0aXR1ZGUiOiA0MS44NzgxLCAibG9uZ2l0dWRlIjogLTg3LjYyOTgsICJ0ZW1wZXJhdHVyZSI6IC00LjAxLCAiaHVtaWRpdHkiOiA4Ny45OSwgIndpbmRfc3BlZWQiOiAxNy4yMywgInA= (truncated),evnt-storing,0,170,2026-04-17T22:02:10.062Z,0
null,eyJjaXR5IjogIlBhcmlzIiwgImxhdGl0dWRlIjogNDguODU2NiwgImxvbmdpdHVkZSI6IDIuMzUyMiwgInRlbXBlcmF0dXJlIjogNDMuNjEsICJodW1pZGl0eSI6IDYyLjgzLCAid2luZF9zcGVlZCI6IDEuNDcsICJwcmVzc3U= (truncated),evnt-storing,0,171,2026-04-17T22:02:15.284Z,0
null,eyJjaXR5IjogIkNhcGUgVG93biIsICJsYXRpdHVkZSI6IC0zMy45MjQ5LCAibG9uZ2l0dWRlIjogMTguNDI0MSwgInRlbXBlcmF0dXJlIjogLTE1LjI4LCAiaHVtaWRpdHkiOiA1My40OCwgIndpbmRfc3BlZWQiOiAxMS4zNyw= (truncated),evnt-storing,0,172,2026-04-17T22:02:20.519Z,0
null,eyJjaXR5IjogIlNlYXR0bGUiLCAibGF0aXR1ZGUiOiA0Ny42MDYyLCAibG9uZ2l0dWRlIjogLTEyMi4zMzIxLCAidGVtcGVyYXR1cmUiOiAzNC44NCwgImh1bWlkaXR5IjogODYuNTYsICJ3aW5kX3NwZWVkIjogMjYuMDksICI= (truncated),evnt-storing,0,173,2026-04-17T22:02:25.739Z,0
null,eyJjaXR5IjogIk1vc2NvdyIsICJsYXRpdHVkZSI6IDU1Ljc1NTgsICJsb25naXR1ZGUiOiAzNy42MTczLCAidGVtcGVyYXR1cmUiOiAtNi4wMSwgImh1bWlkaXR5IjogODMuNjMsICJ3aW5kX3NwZWVkIjogMTAuMzcsICJwcmU= (truncated),evnt-storing,0,174,2026-04-17T22:02:30.975Z,0
null,eyJjaXR5IjogIkNoaWNhZ28iLCAibGF0aXR1ZGUiOiA0MS44NzgxLCAibG9uZ2l0dWRlIjogLTg3LjYyOTgsICJ0ZW1wZXJhdHVyZSI6IDM3LjgzLCAiaHVtaWRpdHkiOiA3OS4yMiwgIndpbmRfc3BlZWQiOiA5LjE4LCAicHI= (truncated),evnt-storing,0,175,2026-04-17T22:02:36.24Z,0
null,eyJjaXR5IjogIlN5ZG5leSIsICJsYXRpdHVkZSI6IC0zMy44Njg4LCAibG9uZ2l0dWRlIjogMTUxLjIwOTMsICJ0ZW1wZXJhdHVyZSI6IDE5LjM1LCAiaHVtaWRpdHkiOiAxMC4yNSwgIndpbmRfc3BlZWQiOiAyNi42OCwgInA= (truncated),evnt-storing,0,176,2026-04-17T22:02:41.46Z,0
null,eyJjaXR5IjogIlNhbiBGcmFuY2lzY28iLCAibGF0aXR1ZGUiOiAzNy43NzQ5LCAibG9uZ2l0dWRlIjogLTEyMi40MTk0LCAidGVtcGVyYXR1cmUiOiAyMi4wNiwgImh1bWlkaXR5IjogOC44OCwgIndpbmRfc3BlZWQiOiAxNS4= (truncated),evnt-storing,0,177,2026-04-17T22:02:46.694Z,0
null,eyJjaXR5IjogIlNcdTAwZTNvIFBhdWxvIiwgImxhdGl0dWRlIjogLTIzLjU1MDUsICJsb25naXR1ZGUiOiAtNDYuNjMzMywgInRlbXBlcmF0dXJlIjogMzkuMDQsICJodW1pZGl0eSI6IDI0LjEyLCAid2luZF9zcGVlZCI6IDQ= (truncated),evnt-storing,0,178,2026-04-17T22:02:51.992Z,0
null,eyJjaXR5IjogIlNhbiBGcmFuY2lzY28iLCAibGF0aXR1ZGUiOiAzNy43NzQ5LCAibG9uZ2l0dWRlIjogLTEyMi40MTk0LCAidGVtcGVyYXR1cmUiOiAyMS43MiwgImh1bWlkaXR5IjogNTQuMTcsICJ3aW5kX3NwZWVkIjogNDA= (truncated),evnt-storing,0,179,2026-04-17T22:02:57.243Z,0


In [0]:
# Define the JSON schema
json_schema = StructType([
    StructField("city", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("weather_condition", StringType(), True),
    StructField("timestamp", StringType(), True)
])

In [0]:
# Reading streaming data from the Delta table
silver_df = spark.readStream\
    .format("delta")\
    .table("weather_data_stream.bronze.bronze_data")\
    .withColumn("body", col("value").cast("string"))\
    .withColumn("body", from_json(col("body"), json_schema))



# Casting all columns to string
df_string = silver_df.select(
    col("body.city").cast("string").alias("city"),
    col("body.latitude").cast("string").alias("latitude"),
    col("body.longitude").cast("string").alias("longitude"),
    col("body.temperature").cast("string").alias("temperature"),
    col("body.humidity").cast("string").alias("humidity"),
    col("body.wind_speed").cast("string").alias("wind_speed"),
    col("body.pressure").cast("string").alias("pressure"),
    col("body.precipitation").cast("string").alias("precipitation"),
    col("body.cloud_cover").cast("string").alias("cloud_cover"),
    col("body.weather_condition").cast("string").alias("weather_condition"),
    col("body.timestamp").cast("string").alias("timestamp")
)

# display(df_string)
run_id = int(time.time())
checkpoint_path = f"/Volumes/weather_data_stream/silver/silver_volume/stream/path_{run_id}"

# Writing the streaming data to the Delta table in append mode
df_string.writeStream\
    .option("checkpointLocation", checkpoint_path ) \
    .outputMode("append")\
    .format("delta")\
    .toTable("weather_data_stream.silver.silver_data")

# dbutils.fs.rm("/Volumes/weathery_test/silver/silver_volume/write_stream/offsets", recurse=True)


In [0]:
%sql
SELECT * FROM weather_data_stream.silver.silver_data;

city,latitude,longitude,temperature,humidity,wind_speed,pressure,precipitation,cloud_cover,weather_condition,timestamp
Chicago,41.8781,-87.6298,-4.01,87.99,17.23,1047.17,36.86,53.39,Snowy,2026-04-17 23:02:09
Paris,48.8566,2.3522,43.61,62.83,1.47,1002.6,49.97,31.46,Cloudy,2026-04-17 23:02:15
Cape Town,-33.9249,18.4241,-15.28,53.48,11.37,996.22,10.09,19.52,Foggy,2026-04-17 23:02:20
Seattle,47.6062,-122.3321,34.84,86.56,26.09,1031.32,13.66,49.81,Hazy,2026-04-17 23:02:25
Moscow,55.7558,37.6173,-6.01,83.63,10.37,1012.72,16.97,81.39,Sunny,2026-04-17 23:02:30
Chicago,41.8781,-87.6298,37.83,79.22,9.18,953.19,24.43,65.52,Hazy,2026-04-17 23:02:36
Sydney,-33.8688,151.2093,19.35,10.25,26.68,978.33,43.67,47.28,Windy,2026-04-17 23:02:41
San Francisco,37.7749,-122.4194,22.06,8.88,15.16,960.32,42.26,94.86,Hazy,2026-04-17 23:02:46
São Paulo,-23.5505,-46.6333,39.04,24.12,47.76,1023.61,43.13,87.8,Cloudy,2026-04-17 23:02:51
San Francisco,37.7749,-122.4194,21.72,54.17,40.72,980.01,13.01,31.82,Sunny,2026-04-17 23:02:57


In [0]:
# Reading streaming data from the Silver table
df_gold = spark.readStream \
    .format("delta") \
    .table("weather_data_stream.silver.silver_data")

# Performing advanced aggregation and rounding to 2 decimal places
df_aggregated = df_gold.groupBy(col("city")).agg(
    round(avg(col("temperature")), 2).alias("avg_temperature"),
    round(avg(col("humidity")), 2).alias("avg_humidity"),
    round(avg(col("wind_speed")), 2).alias("avg_wind_speed"),
    count("*").alias("record_count")
)

# display(df_aggregated)

# Writing the aggregated streaming data to the Delta table in append mode
checkpoint_path = f'/Volumes/weather_data_stream/gold/gold_volume/stream/stream/path_{run_id}'
df_aggregated.writeStream \
    .option("checkpointLocation", checkpoint_path) \
    .outputMode("complete") \
    .format("delta") \
    .toTable("weather_data_stream.gold.gold_data")


# dbutils.fs.rm("/Volumes/weathery_test/gold/gold_volume/write_stream/offsets", recurse=True)

 

In [0]:
%sql

select * from weather_data_stream.gold.gold_data

city,avg_temperature,avg_humidity,avg_wind_speed,record_count
San Francisco,11.13,39.37,22.47,16
Berlin,2.66,26.74,25.16,12
Tokyo,18.56,41.39,24.08,8
Sydney,5.13,48.79,15.8,10
Los Angeles,-8.31,62.44,17.41,9
São Paulo,5.43,52.62,33.69,8
Cape Town,-4.03,27.15,17.92,10
New York,8.81,51.55,28.04,18
Toronto,40.21,70.46,23.48,8
Chicago,16.97,65.65,11.23,10
